In [ ]:
import pathlib as pl
from pprint import pprint

import pywatershed as pws

In [ ]:
output_dir = pl.Path("./08_cascading_flow/output")
domain_dir = pl.Path("../test_data/sagehen_5yr/")

In [ ]:
control_file = domain_dir / "sagehen_no_gw_cascades.control"
control = pws.Control.load_prms(control_file)
control.options["parameter_file"] = (
    domain_dir / control.options["parameter_file"][2:]
)
control.options["input_dir"] = domain_dir
control.options["budget_type"] = None
control.options["netcdf_output_dir"] = output_dir
control.options["netcdf_output_var_names"] = [
    *pws.PRMSRunoffCascadesNoDprst.get_variables(),
    *pws.PRMSSoilzoneCascadesNoDprst.get_variables(),
    *pws.PRMSGroundwaterNoDprst.get_variables(),
]

In [ ]:
parameters = pws.parameters.PrmsParameters.load(
    control.options["parameter_file"]
)

In [ ]:
# add the cascading parameters via pre-processing
parameters = pws.utils.preprocess_cascades.preprocess_cascade_params(
    control, parameters
)

In [ ]:
test_data_dir = pl.Path("../test_data")
domain_dir = test_data_dir / "drb_2yr"
dis_hru = None


process_dict = {
    "solar": pws.PRMSSolarGeometry,
    "atmosphere": pws.PRMSAtmosphere,
    "canopy": pws.PRMSCanopy,
    "snow": pws.PRMSSnow,
    "runoff": pws.PRMSRunoffCascadesNoDprst,
    "soilzone": pws.PRMSSoilzoneCascadesNoDprst,
    "groundwater": pws.PRMSGroundwaterNoDprst,
}

model_dict = {
    "control": control,
    "dis_hru": dis_hru,
    "model_order": list(process_dict.keys()),
}

for kk, vv in process_dict.items():
    model_dict[kk] = {"class": vv, "parameters": parameters, "dis": "dis_hru"}

In [ ]:
pprint(model_dict, sort_dicts=False)

In [ ]:
model = pws.Model(model_dict)

In [ ]:
model.run(netcdf_dir=output_dir, finalize=True)

In [ ]:
import xarray as xr

horton_casc = xr.open_dataarray(output_dir / "hru_horton_cascflow.nc")
sz_casc = xr.open_dataarray(output_dir / "hru_sz_cascadeflow.nc")

In [ ]:
# Set gis_dir to a local directory containing the sagehen domain HRU
# shapefile (HRUs.shp); these GIS data are not distributed with pywatershed.
gis_dir = "path/to/sagehen_pws_domain/GIS/"
proc_plot = pws.analysis.process_plot.ProcessPlot(
    gis_dir,
    hru_shp_file_name="HRUs.shp",
    seg_shp_file_name=None,
)
proc_plot.plot_hru_var(
    var_name="hru_horton_cascflow",
    process=model.processes["runoff"],
    # data=horton_casc.mean(dim="time"),
    # data=horton_casc,  # make this a move in the underlying code
    data_units=horton_casc.attrs["units"],
    nhm_id=horton_casc["nhm_id"],
    clim=(0.0, 1.0e-2),
)

In [ ]:
var = "horton_casc"
proc_plot.plot_hru_var(
    var_name=var,
    process=model.processes["runoff"],
    # data=horton_casc.mean(dim="time"),
    data=sz_casc[30, :],
    data_units=horton_casc.attrs["units"],
    nhm_id=horton_casc["nhm_id"],
)

In [ ]:
var = "horton_casc"
proc_plot.plot_hru_var(
    var_name=var,
    process=model.processes["runoff"],
    # data=horton_casc.mean(dim="time"),
    data=sz_casc[135, :],
    data_units=horton_casc.attrs["units"],
    nhm_id=horton_casc["nhm_id"],
)